# Stage 03 — Replication

Reproduce the paper's main OLS/IV regressions. Write JSON + LaTeX table.
Follow `skills/stage_03.md` for detailed instructions.

In [ ]:
from paths import *
import json
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

df   = pd.read_parquet(str(DATASET_PATH))
spec = json.loads(PAPER_SPEC.read_text())

outcome   = spec['identification']['outcome_variable']
treatment = spec['identification']['treatment_variable']
instrument = spec['identification'].get('instrument')
controls  = spec['identification']['controls']
id_type   = spec['identification']['type']

print(f'Identification: {id_type}')
print(f'N obs (complete): {df[[outcome, treatment]].dropna().shape[0]}')

In [ ]:
# --- AGENT: run replications ---
# For each spec in spec['main_results'], run the regression.
# Match controls, fixed effects, and SE correction exactly.
#
# For IV: use linearmodels.IV2SLS
# For OLS with spatial SE: use spreg or manually implement Conley SE
#
# Store results in the list below following the schema in skills/stage_03.md

replication_specs = []

# Example OLS:
# exog = sm.add_constant(df[controls + [treatment]].dropna())
# endog = df.loc[exog.index, outcome]
# model = sm.OLS(endog, exog).fit(cov_type='HC3')
# coef = model.params[treatment]
# se   = model.bse[treatment]

pass

In [ ]:
# --- Compare to published results ---
for r in replication_specs:
    pub   = r['published_coef']
    rep   = r['replicated_coef']
    rdiff = abs(rep - pub) / abs(pub) * 100 if pub != 0 else float('nan')
    r['rel_diff_pct'] = round(rdiff, 2)
    r['match'] = rdiff < 5.0
    print(f"{r['spec']:30s}  pub={pub:.4f}  rep={rep:.4f}  diff={rdiff:.1f}%  {'✓' if r['match'] else '✗'}")

In [ ]:
# --- Write replication_results.json ---
n_pass = sum(r['match'] for r in replication_specs)
replication_results = {
    'specs': replication_specs,
    'overall_pass': n_pass == len(replication_specs),
    'n_specs': len(replication_specs),
    'n_pass': n_pass
}
REPLICATION_RESULTS.write_text(json.dumps(replication_results, indent=2))

worst = max((r['rel_diff_pct'] for r in replication_specs), default=0)
replication_check = {
    'pass': replication_results['overall_pass'],
    'summary': f"{n_pass}/{len(replication_specs)} specs replicated within 5%",
    'worst_rel_diff_pct': worst
}
REPLICATION_CHECK.write_text(json.dumps(replication_check, indent=2))
print(f'✓ {REPLICATION_RESULTS}')
print(f'✓ {REPLICATION_CHECK}')

In [ ]:
# --- AGENT: write LaTeX table ---
# Build a regression table comparing published vs replicated.
# Save to TABLES_DIR / 'table_replication.tex'
# Use standard tabular format with booktabs-style rules.
pass